# TensorRT Edge Deployment: TCN Activity Classifier

Builds a self-contained **`edge_deploy/`** folder to copy to a Jetson / other NVIDIA edge device:

| file | made by | purpose |
|---|---|---|
| `model_def.py` | this notebook | the TCN class + constants (same code as `AISE4010_Project_Group8.ipynb`) |
| `tcn_weights.pt` | this notebook | trained weights + the training-set standardisation (`mu`, `sigma`) |
| `tcn.onnx` | this notebook | model exported for TensorRT (dynamic batch axis) |
| `sample_windows.npz` | this notebook | 2,000 raw windows from the true test set (subjects held out of all training, tuning and model selection) with labels, so the device needs no dataset |
| `compile_trt.py` | this notebook | **run on the device**: ONNX -> TensorRT engine, tuned for *fast compilation* |
| `infer.py` | this notebook | **run on the device**: PyTorch vs TensorRT accuracy, latency, agreement |
| `setup_env.sh` | this notebook | **run on the device first**: creates a venv and installs the pinned dependencies |

**Model.** The notebook's best model is the **untuned TCN** (default `TCN()`: 32 channels, kernel 5, dropout 0.1, 38,630 parameters;
pseudo-test accuracy 0.848 / macro-F1 0.823). The grid-tuned TCN scored worse on the pseudo-test subjects, so it is not used.
The original notebook never saved its weights, so this notebook retrains the model with the same settings (seed 42, Adam lr 1e-3,
batch 256, early stopping on validation loss, stride-1 windows). The score on the true test set can differ noticeably from the original run.

**Why the engine is built on the device.** A TensorRT engine is tied to the GPU and TensorRT version that built it, so only the ONNX file travels; the compile script runs on the target.

In [1]:
import copy
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score

OUT = Path("edge_deploy")                       # everything for the edge device goes here
OUT.mkdir(exist_ok=True)
sys.path.insert(0, str(OUT.resolve()))

SPLIT_ROOT = Path("A_DeviceMotion_data_split/train_split")   # train / valid split written by the main notebook
TEST_DIR = Path("A_DeviceMotion_data_split/test")           # the TRUE test set: outer split, never used for training or tuning
RETRAIN = False        # True: retrain even if edge_deploy/tcn_weights.pt exists
N_SAMPLE = 2000        # test windows bundled for the device
ONNX_OPSET = 17

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training device:", DEVICE)

# ONNX export / verification helpers (only needed here, not on the edge device)
try:
    import onnx, onnxruntime
except ImportError as e:
    raise ImportError("This notebook needs onnx and onnxruntime in its kernel environment: run "
                      "'python -m pip install onnx onnxruntime' in a terminal, then restart the kernel.") from e

Training device: cuda


## Shared Model Definition

Written to `edge_deploy/model_def.py` and imported by this notebook as well, so the notebook and the device run the exact same class.

In [2]:
%%writefile edge_deploy/model_def.py
"""Model definition shared by the notebook, compile_trt.py and infer.py (copied from AISE4010_Project_Group8.ipynb)."""
import torch
import torch.nn as nn

FS = 50               # DeviceMotion sampling rate (Hz)
WINDOW = 100          # 2.0 s of history at 50 Hz
CLASSES = ["dws", "jog", "sit", "std", "ups", "wlk"]
CHANNELS = (["attitude.roll", "attitude.pitch", "attitude.yaw"]
            + [f"{sensor}.{axis}" for sensor in ("gravity", "rotationRate", "userAcceleration") for axis in "xyz"])
N_CLASSES, N_CHANNELS = len(CLASSES), len(CHANNELS)

# Best model of the notebook (untuned TCN: pseudo-test accuracy 0.848, macro-F1 0.823) = the default TCN() settings.
BEST_CONFIG = {"channels": 32, "kernel": 5, "dropout": 0.1}


class CausalBlock(nn.Module):
    def __init__(self, c_in, c_out, kernel, dilation, dropout):
        super().__init__()
        pad = (kernel - 1) * dilation                   # left padding only -> causal
        self.net = nn.Sequential(
            nn.ConstantPad1d((pad, 0), 0.0), nn.Conv1d(c_in, c_out, kernel, dilation=dilation), nn.ReLU(), nn.Dropout(dropout),
            nn.ConstantPad1d((pad, 0), 0.0), nn.Conv1d(c_out, c_out, kernel, dilation=dilation), nn.ReLU(), nn.Dropout(dropout),
        )
        self.residual = nn.Conv1d(c_in, c_out, 1) if c_in != c_out else nn.Identity()

    def forward(self, x):
        return torch.relu(self.net(x) + self.residual(x))


def tcn_receptive_field(kernel, levels):
    return 1 + 2 * (kernel - 1) * (2 ** levels - 1)


class TCN(nn.Module):
    def __init__(self, channels=32, kernel=5, levels=None, dropout=0.1):
        super().__init__()
        if levels is None:                              # smallest depth whose receptive field covers the window
            levels = next(L for L in range(1, 12) if tcn_receptive_field(kernel, L) >= WINDOW)
        self.receptive_field = tcn_receptive_field(kernel, levels)
        self.blocks = nn.Sequential(*[
            CausalBlock(N_CHANNELS if i == 0 else channels, channels, kernel, 2 ** i, dropout) for i in range(levels)
        ])
        self.head = nn.Linear(channels, N_CLASSES)

    def forward(self, x):          # x: (batch, N_CHANNELS, WINDOW), already standardised
        return self.head(self.blocks(x)[:, :, -1])      # features at the last (current) time step


def load_checkpoint(path, device="cpu"):
    """-> (eval-mode model, mu (C,1), sigma (C,1) as float32 numpy) from the file written by the notebook."""
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    model = TCN(**ckpt["config"])
    model.load_state_dict(ckpt["state_dict"])
    return model.to(device).eval(), ckpt["mu"], ckpt["sigma"]

Overwriting edge_deploy/model_def.py


## Data and Windows

Same windowing as the main notebook: one causal 100-sample window per timestep, labelled with the activity at the last sample, first sample repeated at the start of each recording.

In [3]:
from model_def import (BEST_CONFIG, CHANNELS, CLASSES, FS, N_CHANNELS, N_CLASSES, TCN, WINDOW, load_checkpoint)


def load_split(split_dir):
    """Load every <trial>/sub_<n>.csv in a split folder -> {(trial, subject): DataFrame}."""
    recordings = {}
    for f in sorted(Path(split_dir).glob("*/sub_*.csv"), key=lambda p: (p.parent.name, int(p.stem.split("_")[1]))):
        recordings[(f.parent.name, int(f.stem.split("_")[1]))] = pd.read_csv(f, index_col=0)
    if not recordings:
        raise FileNotFoundError(f"No <trial>/sub_*.csv files in {split_dir}; run the main notebook's splitting cell first")
    return recordings


def make_windows(recordings, stride=1):
    X, y = [], []
    for (trial, _), df in recordings.items():
        x = df[CHANNELS].to_numpy(dtype=np.float32).T
        x = np.pad(x, ((0, 0), (WINDOW - 1, 0)), mode="edge")
        w = np.lib.stride_tricks.sliding_window_view(x, WINDOW, axis=1)[:, ::stride]
        X.append(w.transpose(1, 0, 2))
        y.append(np.full(w.shape[1], CLASSES.index(trial.split("_")[0]), dtype=np.int64))
    return np.ascontiguousarray(np.concatenate(X)), np.concatenate(y)


X_test, y_test = make_windows(load_split(TEST_DIR))
print(f"Test windows: {len(X_test):,}")

Test windows: 285,674


## Train (or Load) the Best Model

Mirrors `train_model` in the main notebook. Skipped when `edge_deploy/tcn_weights.pt` already exists (set `RETRAIN = True` to force it).

In [4]:
CKPT = OUT / "tcn_weights.pt"


def to_device(X, mu, sigma):
    """Standardise with the training statistics and keep the windows on DEVICE."""
    t = torch.from_numpy(X).to(DEVICE)
    return t.sub_(torch.from_numpy(mu).to(DEVICE)).div_(torch.from_numpy(sigma).to(DEVICE))


def batches(X, y, batch_size, shuffle):
    idx = torch.randperm(len(X), device=DEVICE) if shuffle else torch.arange(len(X), device=DEVICE)
    for i in range(0, len(X), batch_size):
        j = idx[i:i + batch_size]
        yield X[j], y[j]


def train_tcn(max_epochs=30, patience=4, lr=1e-3, batch_size=256):
    X_train, y_train = make_windows(load_split(SPLIT_ROOT / "train"))
    X_val, y_val = make_windows(load_split(SPLIT_ROOT / "valid"))
    mu = X_train.mean(axis=(0, 2))[:, None]
    sigma = X_train.std(axis=(0, 2))[:, None]
    Xtr, ytr = to_device(X_train, mu, sigma), torch.from_numpy(y_train).to(DEVICE)
    Xva, yva = to_device(X_val, mu, sigma), torch.from_numpy(y_val).to(DEVICE)
    del X_train, X_val
    print(f"Train windows: {len(Xtr):,} | Valid: {len(Xva):,}")

    torch.manual_seed(SEED)
    np.random.seed(SEED)
    model = TCN(**BEST_CONFIG).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    best_val, best_state, wait = np.inf, None, 0
    for epoch in range(1, max_epochs + 1):
        t0 = time.perf_counter()
        model.train()
        tr_loss = torch.zeros((), device=DEVICE)
        for xb, yb in batches(Xtr, ytr, batch_size, shuffle=True):
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()
            tr_loss += loss.detach() * len(xb)
        model.eval()
        val_loss = torch.zeros((), device=DEVICE)
        val_ok = torch.zeros((), device=DEVICE)
        with torch.no_grad():
            for xb, yb in batches(Xva, yva, 4096, shuffle=False):
                logits = model(xb)
                val_loss += loss_fn(logits, yb) * len(xb)
                val_ok += (logits.argmax(1) == yb).sum()
        val_loss = val_loss.item() / len(Xva)
        print(f"epoch {epoch:2d}  train {tr_loss.item() / len(Xtr):.4f}  val {val_loss:.4f}  "
              f"val acc {val_ok.item() / len(Xva):.3f}  ({time.perf_counter() - t0:.0f} s)")
        if val_loss < best_val - 1e-5:
            best_val, wait = val_loss, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience:
                break
    torch.save({"state_dict": best_state, "config": BEST_CONFIG, "mu": mu, "sigma": sigma}, CKPT)
    del Xtr, Xva
    torch.cuda.empty_cache()


if RETRAIN or not CKPT.exists():
    train_tcn()
model, MU, SIGMA = load_checkpoint(CKPT, DEVICE)
print(f"Loaded {CKPT} | config {BEST_CONFIG} | {sum(p.numel() for p in model.parameters()):,} parameters")

Train windows: 755,548 | Valid: 236,886


epoch  1  train 0.0900  val 0.6628  val acc 0.833  (30 s)


epoch  2  train 0.0190  val 1.1011  val acc 0.819  (31 s)


epoch  3  train 0.0112  val 1.3934  val acc 0.811  (30 s)


epoch  4  train 0.0082  val 1.1085  val acc 0.875  (29 s)


epoch  5  train 0.0063  val 1.5178  val acc 0.830  (29 s)
Loaded edge_deploy\tcn_weights.pt | config {'channels': 32, 'kernel': 5, 'dropout': 0.1} | 38,630 parameters


### Score on the True Test Set

The main notebook never scored the true test set; its pseudo-test subjects gave accuracy 0.848 / macro-F1 0.823 for this model (single seed, with large run-to-run variance), so this number is the first honest estimate.

In [5]:
@torch.no_grad()
def predict(model, X, batch_size=4096):
    Xs = ((X - MU) / SIGMA).astype(np.float32)
    return np.concatenate([model(torch.from_numpy(Xs[i:i + batch_size]).to(DEVICE)).argmax(1).cpu().numpy()
                           for i in range(0, len(Xs), batch_size)])


p_test = predict(model, X_test)
print(f"PyTorch true test set: accuracy {accuracy_score(y_test, p_test):.4f} | macro-F1 {f1_score(y_test, p_test, average='macro'):.4f}")

PyTorch true test set: accuracy 0.8302 | macro-F1 0.8210


## Export ONNX and the Sample Windows

The ONNX model takes the **already standardised** window `(batch, 12, 100)` and returns the 6 class logits. `infer.py` does the standardisation with `mu` / `sigma` from the checkpoint.
The batch axis is dynamic in the ONNX file; `compile_trt.py --batch N` fixes it to one size for the engine (fastest to build).

In [6]:
ONNX = OUT / "tcn.onnx"
cpu_model = copy.deepcopy(model).cpu().eval()
torch.onnx.export(cpu_model, torch.zeros(1, N_CHANNELS, WINDOW), str(ONNX),
                  input_names=["window"], output_names=["logits"],
                  dynamic_axes={"window": {0: "batch"}, "logits": {0: "batch"}},
                  opset_version=ONNX_OPSET, dynamo=False)
onnx.checker.check_model(onnx.load(str(ONNX)))

# the exported graph must reproduce PyTorch (checked at a batch size other than the traced one)
x = torch.from_numpy(((X_test[:64] - MU) / SIGMA).astype(np.float32))
with torch.no_grad():
    ref = cpu_model(x).numpy()
sess = onnxruntime.InferenceSession(str(ONNX), providers=["CPUExecutionProvider"])
got = sess.run(None, {"window": x.numpy()})[0]
print(f"{ONNX} ({ONNX.stat().st_size / 1024:.0f} KB) | max |PyTorch - ONNX Runtime| logit diff: {np.abs(ref - got).max():.2e}")
assert np.abs(ref - got).max() < 1e-3 and (ref.argmax(1) == got.argmax(1)).all()

idx = np.linspace(0, len(X_test) - 1, N_SAMPLE).astype(int)       # spread over every recording
np.savez_compressed(OUT / "sample_windows.npz", X=X_test[idx], y=y_test[idx])
print("Sample windows per class:", dict(zip(CLASSES, np.bincount(y_test[idx], minlength=N_CLASSES))))

C:\Users\Andrew_Liang\AppData\Local\Temp\ipykernel_8556\1915970583.py:3: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(cpu_model, torch.zeros(1, N_CHANNELS, WINDOW), str(ONNX),


C:\Users\Andrew_Liang\.conda\envs\aise4010\Lib\site-packages\torch\onnx\_internal\torchscript_exporter\jit_utils.py:308: UserWarning: Constant folding - Only steps=1 can be constant folded for opset >= 10 onnx::Slice op. Constant folding not applied. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\jit\passes\onnx\constant_fold.cpp:180.)
  _C._jit_pass_onnx_node_shape_type_inference(node, params_dict, opset_version)
C:\Users\Andrew_Liang\.conda\envs\aise4010\Lib\site-packages\torch\onnx\_internal\torchscript_exporter\utils.py:712: UserWarning: Constant folding - Only steps=1 can be constant folded for opset >= 10 onnx::Slice op. Constant folding not applied. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\jit\passes\onnx\constant_fold.cpp:180.)
  _C._jit_pass_onnx_graph_shape_type_inference(
C:\Users\Andrew_Liang\.conda\envs\aise4010\Lib\site-packages\torch\onnx\_internal\torchscript_exporter\utils.py:1185: UserWarning: Constant fo

edge_deploy\tcn.onnx (178 KB) | max |PyTorch - ONNX Runtime| logit diff: 4.29e-06


Sample windows per class: {'dws': np.int64(193), 'jog': np.int64(200), 'sit': np.int64(421), 'std': np.int64(445), 'ups': np.int64(237), 'wlk': np.int64(504)}


## `compile_trt.py` (run on the edge device)

Choices that favour **fast compilation** over fastest inference:

- `builder_optimization_level = 0` - the least tactic search TensorRT offers (needs TensorRT >= 8.6; JetPack 6 has it).
- Fixed batch shape (`min = opt = max`), so only one shape is tuned.
- FP32 by default; `--fp16` allows FP16 kernels (usually lower latency on Jetson, at the price of a longer build).
- Small 512 MB workspace, and an on-disk **timing cache** (`tcn.timing_cache`) so rebuilds (e.g. different batch size or after retraining) reuse earlier tactic timings.

In [7]:
%%writefile edge_deploy/compile_trt.py
#!/usr/bin/env python3
"""Compile tcn.onnx into a TensorRT engine ON THE EDGE DEVICE (engines are specific to the GPU / TensorRT version).

Tuned for FAST COMPILATION rather than fastest inference:
  * builder optimisation level 0 (least tactic search; TensorRT >= 8.6)
  * fixed batch shape (min = opt = max), so a single optimisation profile with one shape to tune
  * small workspace, FP32 by default (FP16 adds tactics to try; enable with --fp16 if you want lower latency)
  * on-disk timing cache: rebuilding the same / similar model afterwards reuses earlier tactic timings

Only needs the `tensorrt` python package (JetPack ships it); PyTorch is not required for compiling.

    python compile_trt.py                       # tcn.onnx -> tcn.engine
    python compile_trt.py --fp16 --batch 8
"""
import argparse
import os
import sys
import time

os.environ.setdefault("CUDA_MODULE_LOADING", "LAZY")   # faster CUDA start-up

import tensorrt as trt

HERE = os.path.dirname(os.path.abspath(__file__))


def parse_args():
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--onnx", default=os.path.join(HERE, "tcn.onnx"))
    p.add_argument("--engine", default=os.path.join(HERE, "tcn.engine"))
    p.add_argument("--batch", type=int, default=1, help="fixed batch size baked into the engine (default 1)")
    p.add_argument("--fp16", action="store_true", help="allow FP16 kernels (faster inference, slower build)")
    p.add_argument("--opt-level", type=int, default=0, choices=range(0, 6),
                   help="builder optimisation level 0-5; 0 = fastest build (default), 3 = TensorRT default")
    p.add_argument("--workspace-mb", type=int, default=512, help="max scratch memory for tactics (default 512 MB)")
    p.add_argument("--timing-cache", default=os.path.join(HERE, "tcn.timing_cache"),
                   help="timing cache file, created if missing; pass '' to disable")
    p.add_argument("--verbose", action="store_true")
    return p.parse_args()


def build_engine(args):
    logger = trt.Logger(trt.Logger.VERBOSE if args.verbose else trt.Logger.WARNING)
    builder = trt.Builder(logger)
    # TensorRT 10 networks are always explicit-batch (the flag is deprecated there); TensorRT 8 needs it.
    flags = 0
    if hasattr(trt.NetworkDefinitionCreationFlag, "EXPLICIT_BATCH"):
        flags = 1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH)
    network = builder.create_network(flags)
    parser = trt.OnnxParser(network, logger)
    if not parser.parse_from_file(args.onnx):
        for i in range(parser.num_errors):
            print("ONNX parse error:", parser.get_error(i), file=sys.stderr)
        sys.exit("Failed to parse " + args.onnx)

    config = builder.create_builder_config()
    config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, args.workspace_mb << 20)
    if hasattr(config, "builder_optimization_level"):
        config.builder_optimization_level = args.opt_level
    else:
        print("WARNING: this TensorRT has no builder_optimization_level (needs >= 8.6); using its default effort")

    if args.fp16:
        if not hasattr(trt.BuilderFlag, "FP16"):                    # removed in TensorRT 11 (strongly typed networks only)
            print("WARNING: this TensorRT has no BuilderFlag.FP16 (TensorRT 11); building FP32")
        elif getattr(builder, "platform_has_fast_fp16", True):
            config.set_flag(trt.BuilderFlag.FP16)
        else:
            print("WARNING: this GPU has no fast FP16; building FP32")

    # Pin the (dynamic) batch dimension of the ONNX model to a single shape.
    inp = network.get_input(0)
    if any(d < 0 for d in inp.shape):
        shape = (args.batch,) + tuple(inp.shape)[1:]
        profile = builder.create_optimization_profile()
        profile.set_shape(inp.name, shape, shape, shape)
        config.add_optimization_profile(profile)
    print(f"TensorRT {trt.__version__} | input '{inp.name}' {tuple(inp.shape)} -> batch {args.batch} "
          f"| opt level {args.opt_level} | {'FP16' if args.fp16 else 'FP32'}")

    cache = None
    if args.timing_cache:
        blob = open(args.timing_cache, "rb").read() if os.path.exists(args.timing_cache) else b""
        cache = config.create_timing_cache(blob)
        if not config.set_timing_cache(cache, False):      # cache from another GPU / TensorRT: start empty
            cache = config.create_timing_cache(b"")
            config.set_timing_cache(cache, False)

    t0 = time.perf_counter()
    serialized = builder.build_serialized_network(network, config)
    build_s = time.perf_counter() - t0
    if serialized is None:
        sys.exit("Engine build failed (re-run with --verbose)")

    with open(args.engine, "wb") as f:
        f.write(serialized)
    if cache is not None:
        with open(args.timing_cache, "wb") as f:
            f.write(memoryview(cache.serialize()))
    print(f"Built {args.engine} ({memoryview(serialized).nbytes / 1024:.0f} KB) in {build_s:.1f} s")


if __name__ == "__main__":
    build_engine(parse_args())

Overwriting edge_deploy/compile_trt.py


## `infer.py` (run on the edge device)

Loads the regular PyTorch model and the TensorRT engine, runs both over the bundled windows and prints accuracy, macro-F1, latency (median / p95 / mean per batch, input already on the GPU), throughput, prediction agreement, and the maximum logit difference.
PyTorch tensors serve as the TensorRT I/O buffers, so no `pycuda` is needed.

In [8]:
%%writefile edge_deploy/infer.py
#!/usr/bin/env python3
"""Run the regular PyTorch TCN and the compiled TensorRT engine on the same windows and compare them.

Needs: torch (with CUDA, as shipped for Jetson) and tensorrt. Reports, per backend:
  * accuracy / macro-F1 on the bundled test windows (sample_windows.npz)
  * single-batch latency (median / p95 / mean, input already on the GPU) and throughput
and, between the two: prediction agreement and the largest logit difference.

    python infer.py                         # PyTorch (GPU) vs TensorRT
    python infer.py --cpu --runs 500        # also time PyTorch on the CPU
"""
import argparse
import json
import os
import time

os.environ.setdefault("CUDA_MODULE_LOADING", "LAZY")

import numpy as np
import torch

from model_def import CLASSES, N_CHANNELS, WINDOW, load_checkpoint

HERE = os.path.dirname(os.path.abspath(__file__))


class TRTRunner:
    """Callable wrapper around a serialized engine; uses torch CUDA tensors as I/O buffers (no pycuda needed)."""

    def __init__(self, engine_path):
        import tensorrt as trt
        logger = trt.Logger(trt.Logger.WARNING)
        with open(engine_path, "rb") as f:
            self.engine = trt.Runtime(logger).deserialize_cuda_engine(f.read())
        if self.engine is None:
            raise SystemExit(f"Could not load {engine_path} - was it built on this device with this TensorRT version?")
        self.ctx = self.engine.create_execution_context()
        names = [self.engine.get_tensor_name(i) for i in range(self.engine.num_io_tensors)]
        self.in_name = next(n for n in names if self.engine.get_tensor_mode(n) == trt.TensorIOMode.INPUT)
        self.out_name = next(n for n in names if self.engine.get_tensor_mode(n) == trt.TensorIOMode.OUTPUT)

        shape = tuple(self.engine.get_tensor_shape(self.in_name))
        if min(shape) < 0:                                  # profile-based engine: use its (fixed) batch
            shape = tuple(self.engine.get_tensor_profile_shape(self.in_name, 0)[2])
            self.ctx.set_input_shape(self.in_name, shape)
        self.batch = shape[0]
        self.inp = torch.zeros(shape, dtype=torch.float32, device="cuda")
        self.out = torch.zeros(tuple(self.ctx.get_tensor_shape(self.out_name)), dtype=torch.float32, device="cuda")
        self.ctx.set_tensor_address(self.in_name, self.inp.data_ptr())
        self.ctx.set_tensor_address(self.out_name, self.out.data_ptr())

    def __call__(self, x):
        self.inp.copy_(x)                                   # device-to-device, ordered on the current stream
        self.ctx.execute_async_v3(torch.cuda.current_stream().cuda_stream)
        return self.out


def sync(device):
    if device.type == "cuda":
        torch.cuda.synchronize()


@torch.no_grad()
def latency(fn, x, device, warmup, runs):
    for _ in range(warmup):
        fn(x)
    sync(device)
    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        fn(x)
        sync(device)
        times.append(time.perf_counter() - t0)
    t = np.asarray(times) * 1e3
    return {"median_ms": float(np.median(t)), "p95_ms": float(np.percentile(t, 95)), "mean_ms": float(t.mean())}


@torch.no_grad()
def run_all(fn, X, batch, device):
    """Logits for every window, in fixed-size batches (last batch padded by repeating its final window)."""
    out = []
    for i in range(0, len(X), batch):
        xb = X[i:i + batch]
        n = len(xb)
        if n < batch:
            xb = torch.cat([xb, xb[-1:].expand(batch - n, -1, -1)])
        out.append(fn(xb)[:n].float().clone())
    sync(device)
    return torch.cat(out).cpu().numpy()


def macro_f1(y, p):
    f1s = []
    for c in range(len(CLASSES)):
        tp = np.sum((p == c) & (y == c))
        prec = tp / max(np.sum(p == c), 1)
        rec = tp / max(np.sum(y == c), 1)
        f1s.append(2 * prec * rec / (prec + rec) if prec + rec else 0.0)
    return float(np.mean(f1s))


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--weights", default=os.path.join(HERE, "tcn_weights.pt"))
    ap.add_argument("--engine", default=os.path.join(HERE, "tcn.engine"))
    ap.add_argument("--data", default=os.path.join(HERE, "sample_windows.npz"))
    ap.add_argument("--n-windows", type=int, default=0, help="use only the first N windows (default: all)")
    ap.add_argument("--warmup", type=int, default=50)
    ap.add_argument("--runs", type=int, default=300)
    ap.add_argument("--cpu", action="store_true", help="also time the PyTorch model on the CPU")
    ap.add_argument("--json", default="", help="write the results to this JSON file")
    args = ap.parse_args()

    if not torch.cuda.is_available():
        raise SystemExit("CUDA is not available to PyTorch - TensorRT needs the GPU.")
    gpu = torch.device("cuda")
    trt_model = TRTRunner(args.engine)
    batch = trt_model.batch

    model, mu, sigma = load_checkpoint(args.weights, gpu)
    d = np.load(args.data)
    X_raw, y = d["X"], d["y"]
    if args.n_windows:
        X_raw, y = X_raw[:args.n_windows], y[:args.n_windows]
    X = torch.from_numpy(((X_raw - mu) / sigma).astype(np.float32)).to(gpu)     # same standardisation as training
    print(f"{len(X)} windows ({N_CHANNELS} x {WINDOW}) | engine batch {batch} | GPU: {torch.cuda.get_device_name(0)}")

    backends = {"PyTorch (GPU)": (model, gpu), "TensorRT": (trt_model, gpu)}
    if args.cpu:
        backends["PyTorch (CPU)"] = (load_checkpoint(args.weights, "cpu")[0], torch.device("cpu"))

    rows, logits = {}, {}
    for name, (fn, dev) in backends.items():
        Xd = X.to(dev)
        logits[name] = run_all(fn, Xd, batch, dev)
        pred = logits[name].argmax(1)
        lat = latency(fn, Xd[:batch], dev, args.warmup, args.runs)
        rows[name] = {"accuracy": float((pred == y).mean()), "macro_F1": macro_f1(y, pred), **lat,
                      "windows_per_s": 1e3 * batch / lat["median_ms"]}

    ref, cmp = logits["PyTorch (GPU)"], logits["TensorRT"]
    agree = float((ref.argmax(1) == cmp.argmax(1)).mean())
    max_diff = float(np.abs(ref - cmp).max())

    print(f"\n{'backend':15} {'acc':>7} {'macroF1':>8} {'median ms':>10} {'p95 ms':>8} {'mean ms':>8} {'windows/s':>10}")
    for name, r in rows.items():
        print(f"{name:15} {r['accuracy']:7.4f} {r['macro_F1']:8.4f} {r['median_ms']:10.3f} {r['p95_ms']:8.3f} "
              f"{r['mean_ms']:8.3f} {r['windows_per_s']:10.0f}")
    print(f"\nTensorRT vs PyTorch: {100 * agree:.2f}% identical predictions, max |logit diff| = {max_diff:.2e}")
    print(f"TensorRT speed-up over PyTorch (GPU): {rows['PyTorch (GPU)']['median_ms'] / rows['TensorRT']['median_ms']:.2f}x "
          f"(latency is per batch of {batch}, input already on the GPU)")

    if args.json:
        with open(args.json, "w") as f:
            json.dump({"batch": batch, "agreement": agree, "max_logit_diff": max_diff, "backends": rows}, f, indent=2)


if __name__ == "__main__":
    main()

Overwriting edge_deploy/infer.py


## `setup_env.sh` (run on the edge device)

Creates a virtual environment (Ubuntu 24.04 blocks system-wide `pip install`) and installs everything into it; it ends with a real CUDA kernel test of PyTorch.
- **x86-64:** numpy 1.26.4, torch 2.4.1 and tensorrt 10.3.0 from PyPI.
- **Jetson / any aarch64** (the PyPI `tensorrt` package is an x86-only wheel-stub, and PyPI's aarch64 torch is CPU-only): TensorRT comes from NVIDIA's `apt` repo (JetPack 7.2 / R39 offers 11.4.1), the venv is created with `--system-site-packages` to see it, and a CPU-only torch is replaced by a CUDA build from the `cu130` wheel index (CUDA 13, L4T R38+). `TORCH_INDEX=<url>` overrides the index; for older L4T releases it stops and asks for one.
- The torch CUDA wheel for Jetson/R39 has not been verified by us: the final check will fail loudly if it cannot run a CUDA kernel.
Written with Unix line endings so it runs after being copied from Windows.

In [9]:
SETUP_SH = r'''
#!/usr/bin/env bash
# Create a virtual environment and install everything compile_trt.py / infer.py need, then verify it.
#
#   bash setup_env.sh                                  # venv in ~/trt-venv
#   VENV=/path/to/venv bash setup_env.sh
#   TORCH_INDEX=https://download.pytorch.org/whl/cu130 bash setup_env.sh      # Jetson: force a torch wheel index
#
# Why it looks the way it does (each point was a real failure):
#   * Ubuntu 24.04 refuses system-wide `pip install` (PEP 668)      -> everything goes into a venv, pip runs as `python -m pip`
#   * the PyPI `tensorrt` package is a wheel-stub that only builds on x86-64  -> on Jetson / aarch64 TensorRT comes from apt (NVIDIA's
#     Jetson repo) and the venv is created with --system-site-packages so it can import it
#   * PyPI's aarch64 torch is CPU-only (torch.version.cuda is None)  -> on Jetson a CUDA build is installed from a CUDA wheel index
#
# x86-64 + NVIDIA GPU (driver >= 525): numpy 1.26.4, torch 2.4.1 (CUDA 12.1 wheel), tensorrt 10.3.0 from PyPI / pypi.nvidia.com.
# Jetson / aarch64: TensorRT = whatever the NVIDIA apt repo offers (JetPack 7.2 / R39: 11.4.1), torch = latest CUDA wheel for the release.
set -euo pipefail

VENV="${VENV:-$HOME/trt-venv}"
# slow links (the torch CUDA wheels are large): generous timeouts and retries for every pip call
export PIP_DEFAULT_TIMEOUT=120 PIP_RETRIES=10

say() { printf '\n==> %s\n' "$*"; }
die() { printf '\nERROR: %s\n' "$*" >&2; exit 1; }

ARCH="$(uname -m)"
JETSON=0
if [ -f /etc/nv_tegra_release ] || [ "$ARCH" = "aarch64" ]; then JETSON=1; fi
L4T_MAJOR=""
if [ -f /etc/nv_tegra_release ]; then L4T_MAJOR="$(sed -n 's/^# R\([0-9]*\).*/\1/p' /etc/nv_tegra_release | head -n1)"; fi
say "Python $(python3 --version 2>&1 | cut -d' ' -f2) | $ARCH | $([ "$JETSON" = 1 ] && echo "Jetson/aarch64 (L4T R${L4T_MAJOR:-?})" || echo 'generic Linux x86-64')"

# 1. venv support
if ! python3 -c "import ensurepip, venv" 2>/dev/null; then
    say "Installing python3-full / python3-venv (sudo)"
    sudo apt-get update
    sudo apt-get install -y python3-full python3-venv python3-pip
fi

# 2. Jetson: TensorRT python bindings from the NVIDIA apt repo (must exist BEFORE the venv is checked)
if [ "$JETSON" = 1 ]; then
    if ! /usr/bin/python3 -c "import tensorrt" 2>/dev/null; then
        say "Installing TensorRT python bindings from apt (sudo)"
        sudo apt-get update
        if apt-cache policy python3-libnvinfer | grep -q "Candidate: (none)"; then
            die "apt has no python3-libnvinfer. Is the NVIDIA Jetson repo configured? (ls /etc/apt/sources.list.d/; apt-cache policy | grep -i nvidia)"
        fi
        sudo apt-get install -y python3-libnvinfer python3-libnvinfer-dev
    fi
    /usr/bin/python3 -c "import tensorrt; print('system tensorrt', tensorrt.__version__)" \
        || die "TensorRT installed but not importable by /usr/bin/python3"
fi

# 3. the venv (Jetson: must see the apt-installed tensorrt)
if [ ! -x "$VENV/bin/python" ]; then
    say "Creating virtual environment $VENV"
    if [ "$JETSON" = 1 ]; then python3 -m venv --system-site-packages "$VENV"; else python3 -m venv "$VENV"; fi
else
    say "Re-using virtual environment $VENV"
    if [ "$JETSON" = 1 ] && grep -q "include-system-site-packages = false" "$VENV/pyvenv.cfg"; then
        sed -i 's/include-system-site-packages = false/include-system-site-packages = true/' "$VENV/pyvenv.cfg"
        echo "enabled system site-packages in $VENV/pyvenv.cfg"
    fi
fi
PY="$VENV/bin/python"
# (no `pip install --upgrade pip`: not needed, and it aborted the script on slow / flaky links)

# 4. packages
if [ "$JETSON" = 0 ]; then
    say "Installing numpy 1.26.4, torch 2.4.1, tensorrt 10.3.0 (x86-64)"
    "$PY" -m pip install "numpy==1.26.4" "torch==2.4.1"
    "$PY" -m pip install "tensorrt==10.3.0" --extra-index-url https://pypi.nvidia.com
else
    say "Installing numpy"
    "$PY" -m pip install numpy

    # CUDA wheel index for this L4T release (R38/R39 = JetPack 7 = CUDA 13). Override with TORCH_INDEX=...
    if [ -z "${TORCH_INDEX:-}" ]; then
        if [ -n "$L4T_MAJOR" ] && [ "$L4T_MAJOR" -ge 38 ]; then
            TORCH_INDEX="https://download.pytorch.org/whl/cu130"
        else
            TORCH_INDEX=""
        fi
    fi
    if "$PY" -c "import sys, torch; sys.exit(0 if torch.version.cuda else 1)" 2>/dev/null; then
        say "torch with CUDA already installed: $("$PY" -c 'import torch; print(torch.__version__, torch.version.cuda)')"
    else
        [ -n "$TORCH_INDEX" ] || die "No known CUDA torch wheel index for L4T R${L4T_MAJOR:-?}. Install NVIDIA's CUDA torch wheel for your JetPack into $VENV, or set TORCH_INDEX=<index url>, then re-run."
        if "$PY" -c "import torch" 2>/dev/null; then
            say "Removing the CPU-only torch ($("$PY" -c 'import torch; print(torch.__version__)'))"
            "$PY" -m pip uninstall -y torch
        fi
        say "Installing torch (CUDA) from $TORCH_INDEX"
        "$PY" -m pip install torch --index-url "$TORCH_INDEX"
    fi
fi

# 5. verify
say "Verifying"
"$PY" - <<'EOF'
import sys
import numpy, tensorrt, torch
print("numpy", numpy.__version__, "| torch", torch.__version__, "| torch CUDA", torch.version.cuda, "| tensorrt", tensorrt.__version__)
if torch.version.cuda is None:
    sys.exit("torch is a CPU-only build (torch.version.cuda is None); infer.py needs a CUDA build")
if not torch.cuda.is_available():
    sys.exit("torch cannot see the GPU: wrong CUDA wheel for this JetPack/driver, or the GPU is unsupported by this build (check `nvidia-smi` / `tegrastats`)")
x = torch.ones(8, device="cuda") * 2          # a real CUDA kernel, not just device discovery
assert float(x.sum().item()) == 16.0
print("GPU:", torch.cuda.get_device_name(0))
if int(tensorrt.__version__.split(".")[0]) >= 11:
    print("note: TensorRT 11 - the default FP32 build is expected to work; compile_trt.py --fp16 is untested on it")
EOF

cat <<EOF

Done. Activate the environment, then run the scripts:
    source $VENV/bin/activate
    python compile_trt.py
    python infer.py
EOF
'''
with open(OUT / 'setup_env.sh', 'w', newline=chr(10)) as f:
    f.write(SETUP_SH[1:])
print('wrote', OUT / 'setup_env.sh')

wrote edge_deploy\setup_env.sh


## Check and Bundle

In [10]:
import shutil

archive = shutil.make_archive("edge_deploy", "zip", root_dir=".", base_dir=str(OUT))
print(pd.Series({p.name: f"{p.stat().st_size / 1024:.0f} KB" for p in sorted(OUT.iterdir()) if p.is_file()}).to_string())
print("\nBundle:", archive)

compile_trt.py           5 KB
infer.py                 7 KB
model_def.py             3 KB
sample_windows.npz    8358 KB
setup_env.sh             6 KB
tcn.onnx               178 KB
tcn_weights.pt         158 KB

Bundle: e:\Courses\AISE 4010\Project\edge_deploy.zip


## Running on the Edge Device

### Uploading to the device

From the PC (PowerShell), in the folder that contains `edge_deploy.zip`:

```powershell
cd "<project folder>"
scp edge_deploy.zip <user>@<device-ip>:aise4010_project_group4/
```

- `<device-ip>`: run `hostname -I` on the device. The target folder must already exist (`ssh <user>@<device-ip> mkdir -p aise4010_project_group4`).
- Re-uploading after a rebuild: on the device remove the old copy first so stale files (engine, timing cache) don't linger: `cd ~/aise4010_project_group4 && rm -rf edge_deploy`.

### Running

```bash
cd ~/aise4010_project_group4 && unzip edge_deploy.zip && cd edge_deploy

# 0. One-time setup: creates ~/trt-venv, installs TensorRT / torch (see the notes), then runs a CUDA self-test
bash setup_env.sh                           # VENV=/other/path bash setup_env.sh to change the location
source ~/trt-venv/bin/activate              # every new terminal; prompt shows (trt-venv)

# 1. Compile (only needs tensorrt)
python compile_trt.py                       # tcn.onnx -> tcn.engine, FP32, opt level 0
python compile_trt.py --fp16 --batch 8      # optional: FP16 / different fixed batch size

# 2. Compare the regular and compiled model (needs torch with CUDA + tensorrt)
python infer.py                             # PyTorch (GPU) vs TensorRT
python infer.py --cpu --runs 500            # also time PyTorch on the CPU
```

Without activating: `~/trt-venv/bin/python compile_trt.py`.

Notes:
- The venv is needed because Ubuntu 24.04 blocks system-wide `pip install` (PEP 668). Use `python -m pip`, never `python pip`.
- TensorRT 11 (JetPack 7.2): the default FP32 build is what was designed for; `--fp16` relies on a flag TensorRT 11 may have removed, in which case the script warns and builds FP32.
- The first compile also fills `tcn.timing_cache`; later compiles are faster. Delete it if the GPU or TensorRT version changes.
- If the script reports an engine load error, the engine was built with a different TensorRT version than the one running `infer.py` - recompile on the device.
- `infer.py` latency excludes host-to-device copies and standardisation; it measures only the model forward pass in each backend.
- FP32 TensorRT should match PyTorch to about 1e-5 in the logits; with `--fp16` expect around 1e-2 and an occasional flipped prediction on borderline windows.